# Bias, Variance and Imbalanced Data
## Overfitting, ROC/AUC, Resampling, SMOTE and Noisy Data on Real Medical Data


### Before You Start
- You already know the models, scaling, splitting and cross-validation from Labs 3-8. This lab does not re-teach them.
- Every task ends with a question about **your own results**. A table or plot without a written interpretation earns no marks.
- Use `SEED = 42` everywhere. **Golden rule:** anything that learns from data (scaler, imputer, resampler, threshold) is learned from the **training data only**.
- Install once if needed: `pip install imbalanced-learn`


---

## Part 0 — Setup and the Data
A hospital wants to flag patients at risk of stroke from routine information. Target: `stroke` (1 = had a stroke, 0 = did not).

**What to do:**
1. Import everything you need (including `imblearn`) and set `SEED = 42`.
2. Load the CSV. Report the shape, the columns with missing values, the value counts of `gender`, and the number of patients with `stroke = 1` and `stroke = 0` (count and %).
3. Clean: remove the identifier column; remove any `gender` category with too few rows to learn or test; apply categorical encoding (`pd.get_dummies`) to the text columns.
4. Split 80/20 with `stratify=y`, `random_state=SEED`. Fill missing numeric values with a median learned from the **train** set only. Make a scaled copy (`StandardScaler` fitted on train) for Logistic Regression.
5. Report the number of stroke cases in the train and in the test set.

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    learning_curve, cross_val_predict
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler


SEED = 42


In [20]:
df = pd.read_csv('healthcare-dataset-stroke-data.csv')
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nGender counts:\n{df['gender'].value_counts()}")

stroke_counts = df['stroke'].value_counts()
stroke_pct = df['stroke'].value_counts(normalize=True) * 100
print(f"\nStroke = 1: {stroke_counts.get(1, 0)} ({stroke_pct.get(1, 0):.2f}%)")
print(f"Stroke = 0: {stroke_counts.get(0, 0)} ({stroke_pct.get(0, 0):.2f}%)")

Shape: (5110, 12)

Missing values:
bmi    201
dtype: int64

Gender counts:
gender
Female    2994
Male      2115
Other        1
Name: count, dtype: int64

Stroke = 1: 249 (4.87%)
Stroke = 0: 4861 (95.13%)


In [21]:
df = df.drop(columns=['id'], errors='ignore')
df = df[df['gender'] != 'Other']

X = pd.get_dummies(df.drop(columns=['stroke']), drop_first=True)
y = df['stroke']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imputed), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=X_test.columns, index=X_test.index)

print(f"\nStroke cases in Train: {y_train.sum()}")
print(f"Stroke cases in Test: {y_test.sum()}")


Stroke cases in Train: 199
Stroke cases in Test: 50


**Question 0.1:** In 2–3 lines: how imbalanced is the data? Why did we use `stratify=y`, and what could go wrong with only a few stroke cases in the test set?

**Your answer:**

The data is severely imbalanced, with strokes making up less than 5% of the total cases. We used stratify=y to ensure this 5% ratio is identically preserved in both the training and testing sets. If we didn't stratify, a random split might assign almost zero stroke cases to the test set, making it statistically impossible to evaluate how well the model actually detects strokes.


---

## Part 1 — The Accuracy Trap
**Objective:** show that a model can have excellent accuracy and still be useless.
1. Build a baseline that always predicts the majority class (`DummyClassifier(strategy='most_frequent')`). Report accuracy.
2. Train a default Logistic Regression on the scaled training data. Report Accuracy, Precision, Recall, F1, ROC AUC and PR AUC (average precision) on the test set, and print the confusion matrix.
3. Write a helper function that takes a model and train/test data and returns these six metrics (you will reuse it all lab).

In [22]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    return {
        'Accuracy': round(accuracy_score(y_te, y_pred), 4),
        'Precision': round(precision_score(y_te, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_te, y_pred, zero_division=0), 4),
        'F1': round(f1_score(y_te, y_pred, zero_division=0), 4),
        'ROC AUC': round(roc_auc_score(y_te, y_prob), 4),
        'PR AUC': round(average_precision_score(y_te, y_prob), 4)
    }

dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)
dummy.fit(X_train_scaled, y_train)
dummy_acc = accuracy_score(y_test, dummy.predict(X_test_scaled))
print(f"Dummy Baseline Accuracy: {dummy_acc:.4f}")


Dummy Baseline Accuracy: 0.9511


In [23]:
lr = LogisticRegression(random_state=SEED)
lr_metrics = evaluate_model(lr, X_train_scaled, y_train, X_test_scaled, y_test)

print("Logistic Regression Test Metrics:")
for k, v in lr_metrics.items():
    print(f"  {k}: {v}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr.predict(X_test_scaled)))

Logistic Regression Test Metrics:
  Accuracy: 0.9521
  Precision: 1.0
  Recall: 0.02
  F1: 0.0392
  ROC AUC: 0.8394
  PR AUC: 0.2692

Confusion Matrix:
[[972   0]
 [ 49   1]]


**Question 1.1:** Compare the accuracy of the baseline and of Logistic Regression. What do you conclude?

**Question 1.2:** Look at the confusion matrix and the recall. In the hospital's situation, what does a false negative mean, and which numbers should we trust instead of accuracy?

**Your answer:**

1.1: Both the Dummy Classifier and Logistic Regression achieve roughly 95.1% accuracy (0.9511 vs 0.9521). This proves that accuracy is completely misleading on imbalanced data—a model can look 95% accurate just by guessing "no stroke" for everyone while failing to catch almost all actual strokes.  

1.2: A false negative (49 patients in the confusion matrix) means sending a high-risk patient home telling them they are fine, leading to an unprevented stroke. Because our Recall is practically zero (0.0200, catching only 1 out of 50 strokes), we must trust Recall, PR AUC, and ROC AUC instead of accuracy to judge whether the model actually identifies sick patients.   